# SGBA Demo — Solar Geometry-Biased Attention for PV Forecasting

Quick walkthrough of training, evaluation, and statistical testing.

If running on Colab, install the package first:
```python
!pip install -e .
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from sgba.models import SGBA, PatchTST, DLinear
from sgba.training import train_model, TrainingConfig
from sgba.data.datasets import make_dataloaders
from sgba.training.metrics import diebold_mariano_test, block_bootstrap_ci
from sgba.utils.seeds import set_seed
from scripts._common import make_synthetic_dataset, split_chronological

set_seed(42)

## 1. Build a small synthetic dataset
Replace with real DKASC / PVGIS data in production.

In [ ]:
df = make_synthetic_dataset(n_days=200, seed=42)
df_train, df_val, df_test = split_chronological(df)
print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

train_loader, val_loader, test_loader, meta = make_dataloaders(
    df_train, df_val, df_test, site='dkasc_site7',
    config=TrainingConfig(batch_size=64, epochs=10),  # short for demo
)
print(f"elev_mean={meta['elev_mean']:.2f}, elev_std={meta['elev_std']:.2f}")

## 2. Train SGBA-Elev (headline model)

In [ ]:
model = SGBA(
    geometry_mode='elev',
    elev_mean=meta['elev_mean'],
    elev_std=meta['elev_std'],
)
print(f'Parameters: {model.count_parameters():,}')

results = train_model(model, train_loader, val_loader, test_loader,
                      config=TrainingConfig(epochs=10, batch_size=64),
                      verbose=True)
print(f"\nTest metrics: {results['metrics']}")

## 3. Train baselines for comparison

In [ ]:
baselines = {
    'DLinear': DLinear(),
    'PatchTST': PatchTST(),
}
baseline_results = {}
for name, m in baselines.items():
    r = train_model(m, train_loader, val_loader, test_loader,
                    config=TrainingConfig(epochs=10, batch_size=64))
    baseline_results[name] = r
    print(f"{name:10s}  RMSE={r['metrics']['RMSE']:.4f}  R²={r['metrics']['R2']:.4f}")

## 4. Diebold-Mariano significance test

Tests H0: equal predictive accuracy vs H1: different.
Uses Newey-West HAC standard errors with max_lag=24 (daily cycle).

In [ ]:
import numpy as np

err_sgba = results['errors']
for name, r in baseline_results.items():
    err_b = r['errors']
    dm_stat, p_val = diebold_mariano_test(err_sgba, err_b, max_lag=24)
    mean_diff, ci_lo, ci_hi, _ = block_bootstrap_ci(err_sgba, err_b, n_bootstrap=500)
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    print(f"vs {name:10s}  DM={dm_stat:6.2f}  p={p_val:.4g} {sig}  "
          f"ΔRMSE={mean_diff:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}]")